In [ ]:
import pandas as pd
import re
import os
import glob
import warnings
import plotly.graph_objects as go
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

In [ ]:
import os
import glob
import pandas as pd

# --- CONFIG ---
FOLDER_PATH = r"downloaded_tradeatlas_files"

# --- STEP 0: Rename files that do NOT start with 'TradeAtlas' ---
for file_path in glob.glob(os.path.join(FOLDER_PATH, "*.xlsx")):
    filename = os.path.basename(file_path)
    if not filename.startswith("TradeAtlas"):
        new_filename = "TradeAtlas_" + filename
        new_path = os.path.join(FOLDER_PATH, new_filename)
        os.rename(file_path, new_path)
        print(f"Renamed: {filename} → {new_filename}")

# --- STEP 1: Locate all TradeAtlas Excel files ---
excel_files = glob.glob(os.path.join(FOLDER_PATH, "TradeAtlas*.xlsx"))
print(f"Found {len(excel_files)} TradeAtlas files.")

# --- STEP 2: Read each file with the proper header structure ---
merged_data = []
for file_path in excel_files:
    try:
        # Skip first category row, use second row as header
        df = pd.read_excel(file_path, header=1)

        # Drop empty columns
        df = df.dropna(axis=1, how="all")

        # Normalize column names
        df.columns = (
            df.columns.astype(str)
            .str.strip()
            .str.lower()
            .str.replace(r"[^\w]+", "_", regex=True)
        )

        # Add source file name
        df["source_file"] = os.path.basename(file_path)

        merged_data.append(df)
        print(f"Loaded {os.path.basename(file_path)} with {len(df)} rows")

    except Exception as e:
        print(f"Could not read {file_path}: {e}")

# --- STEP 3: Merge all dataframes ---
if merged_data:
    merged_df = pd.concat(merged_data, ignore_index=True)
else:
    raise ValueError("No valid TradeAtlas files were loaded.")

# --- STEP 4: Cleanup ---
merged_df = merged_df.loc[:, ~merged_df.columns.duplicated()]

important_cols = [
    "importer_name", "exporter_name", "hs_code", "fob_value",
    "importer_country", "exporter_country", "country_of_origin",
    "arrival_date", "declaration_number", "gross_weight", "data_type"
]

cols_sorted = [c for c in important_cols if c in merged_df.columns] + \
              [c for c in merged_df.columns if c not in important_cols]

merged_df = merged_df[cols_sorted]

merged_df.head()


In [ ]:
merged_df["importer_name"].value_counts()

In [ ]:
OUTPUT_FILE = r"outputs/merged_tradeatlas_clean.xlsx"
merged_df.to_excel(OUTPUT_FILE, index=False)